# RQ2: few-shot museum fine-tuning (Met → Rijks and Rijks → Met)

Before you start: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (about 45–60 minutes).

**Question:** if we give the WikiArt model a few labeled paintings from one museum (5, 10, 25 or 50 per style), does it get better on the *other* museum?

What this does:
1. Picks the few-shot paintings from the source museum (same picks for every model, so comparisons are paired; the 5 are inside the 10, the 10 inside the 25, and so on).
2. Builds one **fixed test set** per direction from the target museum: only painters who are **not** in WikiArt train (as in RQ1) **and not anywhere in the source museum**, so a painter seen during fine-tuning can never be in the test.
3. **ResNet-50**: starts from your WikiArt checkpoints (seeds 1–3) and fine-tunes for a fixed 200 steps, each batch half museum shots and half WikiArt images (so it does not forget WikiArt). Settings are fixed in advance, nothing is tuned on museum data.
4. **Control:** the same 200 steps with WikiArt images only (`k0ft`). If this also helps, the gain is from extra training, not from museum data.
5. **CLIP and DINOv2 probes** (optional, needs `wikiart_runs/probes/features_*.npz` from the probe notebook): refit the linear classifier on WikiArt train + the shots.
6. Scores everything (balanced accuracy, macro-F1, top-1, top-3, ECE), with bootstrap intervals and the gain over zero-shot (k = 0), and draws the curve.

Uses the same `My Drive/colab_data.zip`. Results go to `My Drive/wikiart_runs/rq2_fewshot/`. If Colab disconnects, just **Run all** again: finished runs are skipped.

## 1. Connect Drive and unpack the data

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/data/final/wikiart_split.csv'):
        !cp /content/drive/MyDrive/colab_data.zip /content/
        !cd /content && unzip -q -o colab_data.zip && rm colab_data.zip
    !nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Settings (no need to change)

In [ ]:
import os, json, copy, time
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path(os.environ.get('RQ2_ROOT', '/content'))
DATA = ROOT / 'data'
RUNS = Path(os.environ.get('RQ2_RUNS', '/content/drive/MyDrive/wikiart_runs'))
OUT = RUNS / 'rq2_fewshot'; OUT.mkdir(parents=True, exist_ok=True)
RES = OUT / 'resnet'; RES.mkdir(exist_ok=True)

SEEDS = [1, 2, 3]
SHOTS = [5, 10, 25, 50]                         # paintings per style from the source museum
DIRECTIONS = [('met', 'rijks'), ('rijks', 'met')]
STEPS, LR, BATCH = 200, 2e-5, 32                # fixed in advance (not tuned on museum data)
REPLAY_PER_STYLE = 700                          # WikiArt train images kept in memory for replay
PROBE_C = {'clip': 0.1, 'dinov2': 0.003}        # chosen on WikiArt val in the probe notebook
B = 2000                                        # bootstrap samples

for s in SEEDS:
    ck = RUNS / f'resnet50_seed{s}' / 'best.pt'
    size = ck.stat().st_size / 1e6 if ck.exists() else 0
    print(f'seed {s}: best.pt {size:.1f} MB', '' if size > 10 else '  <-- MISSING or empty')
    assert size > 10, f'{ck} is missing or empty'

## 3. Data, fixed test sets and few-shot picks

In [ ]:
CLASSES = json.loads((RUNS / 'resnet50_seed1' / 'config.json').read_text())['classes']
split = pd.read_csv(DATA / 'final/wikiart_split.csv')
assert CLASSES == sorted(split['style'].unique()), 'class list differs from the probe notebook'
museum = pd.read_csv(DATA / 'final/museum_test.csv')
museum = museum[museum['style'].isin(CLASSES)]
exists = lambda df: df[df['image'].map(lambda p: (DATA / p).exists())].reset_index(drop=True)
WIKI = {s: exists(split[split['split'] == s]) for s in ('train', 'val', 'test')}
MUS = {k: exists(museum[museum['museum'] == k]) for k in ('met', 'rijks')}
lab = lambda df: np.array([CLASSES.index(s) for s in df['style']])
Y = {k: lab(df) for k, df in {**WIKI, **MUS}.items()}

# One key per painter: Wikidata id if known, else the name. Anonymous / "French Painter" style names get no key.
def artist_key(r):
    q = r.get('artist_wikidata')
    if isinstance(q, str) and q.startswith('Q'):
        return q
    a = r['artist'].strip().lower() if isinstance(r.get('artist'), str) else ''
    if a in ('', 'anonymous', 'anoniem', 'unknown', 'unknown artist') or a.endswith(' painter'):
        return None
    return a
for df in MUS.values():
    df['akey'] = df.apply(artist_key, axis=1)

# Fixed test set per direction: target paintings by painters unseen in WikiArt train and absent from the source museum
TEST = {}
for src, tgt in DIRECTIONS:
    src_artists = set(MUS[src]['akey'].dropna())
    t = MUS[tgt]
    TEST[(src, tgt)] = np.where((t['artist_group'] == 'unseen') & ~t['akey'].isin(src_artists))[0]

# Few-shot picks: per style a random order of the source museum's paintings; k shots = the first k (nested)
ORDER = {}
for src in ('met', 'rijks'):
    for seed in SEEDS:
        rng = np.random.default_rng(100 * seed + (src == 'rijks'))
        ORDER[(src, seed)] = {c: rng.permutation(np.where(Y[src] == c)[0]) for c in range(len(CLASSES))}
def shots(src, seed, k):
    return np.concatenate([o[:k] for o in ORDER[(src, seed)].values()])

rows = []
for src in ('met', 'rijks'):
    for seed in SEEDS:
        for k in SHOTS:
            for i in shots(src, seed, k):
                r = MUS[src].iloc[i]
                rows.append(dict(source=src, seed=seed, k=k, id=r['id'], style=r['style'], artist=r['artist']))
pd.DataFrame(rows).to_csv(OUT / 'fewshot_picks.csv', index=False)

short = [c.split(' ')[0][:9] for c in CLASSES]
print('Fixed test sets (paintings per style):')
for (src, tgt), idx in TEST.items():
    n = np.bincount(Y[tgt][idx], minlength=len(CLASSES))
    print(f'  {src}->{tgt}: n={len(idx):>5} of {len(MUS[tgt])}   ' + '  '.join(f'{s} {v}' for s, v in zip(short, n)))
print('\nShots actually available at k=50 (a style with fewer paintings uses all it has):')
for src in ('met', 'rijks'):
    n = np.bincount(Y[src][shots(src, 1, 50)], minlength=len(CLASSES))
    print(f'  {src:<6}' + '  '.join(f'{s} {v}' for s, v in zip(short, n)))

from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
print('\nFloors on the fixed test sets (chance balanced accuracy is always 16.7):')
for (src, tgt), idx in TEST.items():
    y = Y[tgt][idx]; maj = np.full_like(y, np.bincount(y).argmax())
    print(f'  {src}->{tgt}: majority-class guess  acc {100*accuracy_score(y, maj):.1f}  '
          f'macro-F1 {100*f1_score(y, maj, average="macro", labels=range(len(CLASSES)), zero_division=0):.1f}  bal acc 16.7')

## 4. Load images into memory (about 3 minutes)

In [ ]:
import torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from concurrent.futures import ThreadPoolExecutor

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resize224 = transforms.Resize((224, 224))       # exactly the test transform of train.py

def load_all(paths, size):
    def one(p):
        img = Image.open(DATA / p).convert('RGB')
        img = resize224(img) if size == 224 else img.resize((256, 256))
        return np.asarray(img)
    with ThreadPoolExecutor(8) as ex:
        return torch.from_numpy(np.stack(list(ex.map(one, paths))))

t0 = time.time()
X224 = {k: load_all(df['image'], 224) for k, df in {'test': WIKI['test'], **MUS}.items()}   # for scoring
X256 = {k: load_all(df['image'], 256) for k, df in MUS.items()}                              # shots
rng = np.random.default_rng(0)
REPLAY = np.concatenate([rng.choice(np.where(Y['train'] == c)[0], min(REPLAY_PER_STYLE, (Y['train'] == c).sum()), replace=False)
                         for c in range(len(CLASSES))])
X256['replay'] = load_all(WIKI['train']['image'].values[REPLAY], 256)
Y['replay'] = Y['train'][REPLAY]
print('scoring sets:', {k: len(v) for k, v in X224.items()}, ' shot/replay sets:', {k: len(v) for k, v in X256.items()}, f'  ({(time.time() - t0) / 60:.1f} min)')

## 5. ResNet-50 few-shot fine-tuning (about 30–40 minutes for 3 seeds)
Each finished run is saved to Drive straight away, so a disconnect loses at most one run.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
AMP = dev.type == 'cuda'
aug = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.9, 1.1), antialias=True),
                          transforms.RandomHorizontalFlip()])            # same augmentation as train.py

def to_input(x_uint8):                       # N,H,W,C uint8 -> normalized N,C,H,W on the GPU
    x = x_uint8.to(dev, non_blocking=True).permute(0, 3, 1, 2).float().div(255)
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def predict(model, X, bs=200):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(dev.type, dtype=torch.float16, enabled=AMP):
            out.append(torch.softmax(model(to_input(X[i:i + bs])).float(), 1).cpu())
    return torch.cat(out).numpy()

def base_model(seed):
    m = models.resnet50(); m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    m.load_state_dict(torch.load(RUNS / f'resnet50_seed{seed}' / 'best.pt', map_location='cpu'))
    return m.to(dev).to(memory_format=torch.channels_last)

def batch_from(X, idx):
    return torch.stack([aug(X[i].permute(2, 0, 1)) for i in idx]).permute(0, 2, 3, 1)

def finetune(model, src, shot_idx, seed):
    g = np.random.default_rng(seed); torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    by_class = [np.where(Y['replay'] == c)[0] for c in range(len(CLASSES))]
    n_shot = BATCH // 2 if len(shot_idx) else 0
    model.train()
    for mod in model.modules():               # keep BatchNorm statistics from WikiArt (small, mixed batches)
        if isinstance(mod, nn.BatchNorm2d):
            mod.eval()
    for step in range(STEPS):
        rep = [g.choice(by_class[c]) for c in g.integers(0, len(CLASSES), BATCH - n_shot)]   # class-balanced WikiArt
        xs, ys = [batch_from(X256['replay'], rep)], [Y['replay'][rep]]
        if n_shot:
            sh = g.choice(shot_idx, n_shot)
            xs.append(batch_from(X256[src], sh)); ys.append(Y[src][sh])
        x, y = to_input(torch.cat(xs)), torch.from_numpy(np.concatenate(ys)).to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=AMP):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return model

for seed in SEEDS:
    base = base_model(seed); state = copy.deepcopy(base.state_dict())
    f = RES / f'k0_seed{seed}.npz'
    if not f.exists():
        P = {k: predict(base, X224[k]) for k in ('test', 'met', 'rijks')}
        got = balanced_accuracy_score(Y['test'], P['test'].argmax(1))
        want = json.loads((RUNS / f'resnet50_seed{seed}' / 'metrics.json').read_text())['wikiart_test']['balanced_accuracy']
        print(f'seed {seed} sanity check: WikiArt test bal acc {got:.3f}, metrics.json {want:.3f}')
        assert abs(got - want) < 0.01, 'checkpoint does not match metrics.json: stop and check the run folder'
        np.savez(f, **P)
    jobs = [('k0ft', None, None, 0)] + [(f'{s}2{t}_k{k}', s, t, k) for s, t in DIRECTIONS for k in SHOTS]
    for name, src, tgt, k in jobs:
        f = RES / f'{name}_seed{seed}.npz'
        if f.exists():
            continue
        t0 = time.time()
        base.load_state_dict(state)
        model = finetune(base, src, shots(src, seed, k) if k else np.array([], int), seed)
        sets = ('test', 'met', 'rijks') if k in (0, 50) else (tgt,)          # WikiArt test only at k=0ft and k=50 (forgetting check)
        P = {s: predict(model, X224[s]) for s in sets}
        np.savez(f, **P)
        msg = '  '.join(f'{s} {100*balanced_accuracy_score(Y[s], P[s].argmax(1)):.1f}' for s in sets)
        print(f'seed {seed} {name:<14} bal acc (all paintings): {msg}   ({time.time() - t0:.0f} s)', flush=True)
print('ResNet-50 runs done')

## 6. CLIP and DINOv2 probes with the same shots (optional, about 10–20 minutes)
Skipped automatically if the probe features are not on Drive.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
PROBES = []
for name, C in PROBE_C.items():
    ff = RUNS / 'probes' / f'features_{name}.npz'
    if not ff.exists():
        print(f'{name}: no features on Drive, skipping'); continue
    F = dict(np.load(ff))
    for k, n in (('train', len(WIKI['train'])), ('test', len(WIKI['test'])), ('met', len(MUS['met'])), ('rijks', len(MUS['rijks']))):
        assert len(F[k]) == n, f'{name} {k}: {len(F[k])} features but {n} images (data changed since the probe notebook?)'
    F = {k: v / np.linalg.norm(v, axis=1, keepdims=True) for k, v in F.items()}
    sc = StandardScaler().fit(F['train']); F = {k: sc.transform(v) for k, v in F.items()}
    d = OUT / name; d.mkdir(exist_ok=True); PROBES.append(name)
    t0 = time.time()
    basef = d / 'base_clf.joblib'
    if basef.exists():                      # reuse the WikiArt-only fit
        clf0 = joblib.load(basef)
    else:
        clf0 = LogisticRegression(C=C, max_iter=3000).fit(F['train'], Y['train']); joblib.dump(clf0, basef)
    if not (d / 'k0.npz').exists():
        np.savez(d / 'k0.npz', **{s: clf0.predict_proba(F[s]) for s in ('test', 'met', 'rijks')})
    print(f'{name}: WikiArt-only probe ready ({time.time() - t0:.0f} s)')
    for seed in SEEDS:
        for src, tgt in DIRECTIONS:
            for k in SHOTS:
                f = d / f'{src}2{tgt}_k{k}_seed{seed}.npz'
                if f.exists():
                    continue
                sh = shots(src, seed, k)
                X = np.vstack([F['train'], F[src][sh]]); y = np.concatenate([Y['train'], Y[src][sh]])
                # museum shots and WikiArt get equal total weight (like the 50/50 ResNet batches); total weight unchanged
                w = np.concatenate([np.full(len(F['train']), 0.5), np.full(len(sh), 0.5 * len(F['train']) / len(sh))])
                clf = copy.deepcopy(clf0); clf.set_params(warm_start=True)
                clf.fit(X, y, sample_weight=w)
                np.savez(f, **{s: clf.predict_proba(F[s]) for s in ('test', tgt)})
            print(f'  {name} seed {seed} {src}->{tgt} done ({time.time() - t0:.0f} s)', flush=True)

## 7. Results: few-shot curve with bootstrap intervals
Every number is on the fixed test set (painters never seen in WikiArt train or the source museum), averaged over seeds 1–3. `gain` is the change from k = 0 for the same model, with a paired bootstrap interval and p-value.

In [ ]:
def ece(y, pr, bins=15):
    conf, p = pr.max(1), pr.argmax(1); e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((p[m] == y[m]).mean() - conf[m].mean())
    return e

def load(model, name, seed, key):
    d = RES if model == 'resnet' else OUT / model
    if name == 'k0' and model != 'resnet':
        return np.load(d / 'k0.npz')[key]
    f = d / (f'{name}_seed{seed}.npz')
    return np.load(f)[key] if f.exists() and key in np.load(f) else None

def boot_stats(y, P, counts):
    """bal acc and macro-F1 per bootstrap sample, averaged over seeds. P: list of prob arrays (one per seed)."""
    K = len(CLASSES); Y1 = np.eye(K)[y]; ba, f1 = 0, 0
    for pr in P:
        P1 = np.eye(K)[pr.argmax(1)]
        tp, nt, npred = counts @ (Y1 * P1), counts @ Y1, counts @ P1
        ba = ba + np.nanmean(np.where(nt > 0, tp / np.maximum(nt, 1), np.nan), 1) / len(P)
        f1 = f1 + (2 * tp / np.maximum(nt + npred, 1)).mean(1) / len(P)
    return ba, f1

MODELS = ['resnet'] + PROBES
LABEL = {'resnet': 'ResNet-50 fine-tune', 'clip': 'CLIP probe', 'dinov2': 'DINOv2 probe'}
rows, curves = [], {}
for src, tgt in DIRECTIONS:
    idx = TEST[(src, tgt)]; y = Y[tgt][idx]; n = len(idx)
    g = np.random.default_rng(0)
    counts = np.stack([np.bincount(g.integers(0, n, n), minlength=n) for _ in range(B)]).astype(float)
    for model in MODELS:
        ks = [0, 'k0ft'] + SHOTS if model == 'resnet' else [0] + SHOTS
        ref = None
        for k in ks:
            name = 'k0' if k == 0 else (k if k == 'k0ft' else f'{src}2{tgt}_k{k}')
            seeds = SEEDS if (model == 'resnet' or k != 0) else [SEEDS[0]]
            P_all = [load(model, name, s, tgt) for s in seeds]
            if any(p is None for p in P_all):
                continue
            P = [p[idx] for p in P_all]
            ba_b, f1_b = boot_stats(y, P, counts)
            if k == 0: ref, ref_pt = ba_b, 100 * np.mean([balanced_accuracy_score(y, p.argmax(1)) for p in P])
            per = [dict(ba=balanced_accuracy_score(y, p.argmax(1)), f1=f1_score(y, p.argmax(1), average='macro', labels=range(len(CLASSES)), zero_division=0),
                        acc=accuracy_score(y, p.argmax(1)), top3=(np.argsort(-p, 1)[:, :3] == y[:, None]).any(1).mean(), ece=ece(y, p)) for p in P]
            m = {key: np.mean([d[key] for d in per]) for key in per[0]}
            row = dict(direction=f'{src}->{tgt}', model=LABEL[model], k=k, n_test=n, seeds=len(seeds),
                       bal_acc=100 * m['ba'], bal_acc_sd=100 * np.std([d['ba'] for d in per]),
                       bal_acc_lo=100 * np.percentile(ba_b, 2.5), bal_acc_hi=100 * np.percentile(ba_b, 97.5),
                       macro_f1=100 * m['f1'], f1_lo=100 * np.percentile(f1_b, 2.5), f1_hi=100 * np.percentile(f1_b, 97.5),
                       top1=100 * m['acc'], top3=100 * m['top3'], ece=m['ece'],
                       bal_acc_all_target=100 * np.mean([balanced_accuracy_score(Y[tgt], p.argmax(1)) for p in P_all]))
            if ref is not None and k != 0:
                diff = ba_b - ref
                row.update(gain=row['bal_acc'] - ref_pt,
                           gain_lo=100 * np.percentile(diff, 2.5), gain_hi=100 * np.percentile(diff, 97.5),
                           p=min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean())))
            wt = [load(model, name, s, 'test') for s in seeds]
            if all(w is not None for w in wt):
                row['wikiart_test_bal_acc'] = 100 * np.mean([balanced_accuracy_score(Y['test'], w.argmax(1)) for w in wt])
            row['recall'] = np.mean([[(p.argmax(1)[y == c] == c).mean() if (y == c).any() else np.nan for c in range(len(CLASSES))] for p in P], 0)
            rows.append(row)
            curves.setdefault((src, tgt, model), []).append((k, row))
res = pd.DataFrame(rows)
res.drop(columns='recall').to_csv(OUT / 'rq2_summary.csv', index=False)

lines = []
for d in res['direction'].unique():
    lines.append(f'\n=== {d}  (fixed test set, n = {res[res.direction == d].n_test.iloc[0]}; chance bal acc 16.7) ===')
    lines.append(f'{"model":<21}{"k":>5}{"bal acc [95% CI]":>22}{"SD":>6}{"macro-F1":>10}{"top-1":>7}{"top-3":>7}{"ECE":>7}'
                 f'{"gain vs k=0 [95% CI]":>27}{"p":>7}{"WikiArt":>9}')
    for _, r in res[res.direction == d].iterrows():
        gain = f'{r.gain:+5.1f} [{r.gain_lo:+5.1f}, {r.gain_hi:+5.1f}]' if pd.notna(r.get('gain')) else ''
        pv = f'{r.p:.3f}' if pd.notna(r.get('p')) else ''
        wt = f'{r.wikiart_test_bal_acc:.1f}' if pd.notna(r.get('wikiart_test_bal_acc')) else ''
        lines.append(f'{r.model:<21}{str(r.k):>5}{r.bal_acc:>7.1f} [{r.bal_acc_lo:4.1f}, {r.bal_acc_hi:4.1f}]{r.bal_acc_sd:>6.1f}'
                     f'{r.macro_f1:>10.1f}{r.top1:>7.1f}{r.top3:>7.1f}{r.ece:>7.3f}{gain:>27}{pv:>7}{wt:>9}')
lines.append('\nk0ft = same 200 fine-tuning steps with WikiArt images only (control). WikiArt = WikiArt test bal acc after fine-tuning (forgetting check).')
report = '\n'.join(lines); print(report)
(OUT / 'rq2_report.txt').write_text(report)

print('\nRecall per style (fixed test set, mean over seeds), k = 0 vs k = 50:')
for d in res['direction'].unique():
    for model in res['model'].unique():
        sub = res[(res.direction == d) & (res.model == model) & res.k.isin([0, 50])]
        for _, r in sub.iterrows():
            print(f'  {d:<11}{model:<21}k={str(r.k):<3}' + '  '.join(f'{s} {100*v:4.1f}' for s, v in zip(short, r.recall)))

## 8. The few-shot curve (saved as rq2_curve.png)

In [ ]:
import matplotlib.pyplot as plt
COLOR = {'resnet': '#2a78d6', 'clip': '#eb6834', 'dinov2': '#1baf7a'}
xs = {0: 0, 5: 1, 10: 2, 25: 3, 50: 4}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (src, tgt) in zip(axes, DIRECTIONS):
    for model in MODELS:
        pts = [(xs[k], r) for k, r in curves.get((src, tgt, model), []) if k != 'k0ft']
        if not pts: continue
        x = [p[0] for p in pts]; r = [p[1] for p in pts]
        ax.fill_between(x, [q['bal_acc_lo'] for q in r], [q['bal_acc_hi'] for q in r], color=COLOR[model], alpha=0.12, lw=0)
        ax.plot(x, [q['bal_acc'] for q in r], color=COLOR[model], lw=2, marker='o', ms=6, label=LABEL[model])
        ax.annotate(LABEL[model].split(' ')[0], (x[-1], r[-1]['bal_acc']), xytext=(6, 0), textcoords='offset points',
                    va='center', fontsize=9, color='#52514e')
        ctrl = [q for k, q in curves.get((src, tgt, model), []) if k == 'k0ft']
        if ctrl:
            ax.plot([0], [ctrl[0]['bal_acc']], marker='o', ms=8, mfc='white', mec=COLOR[model], mew=2, ls='none',
                    label='ResNet-50, 200 extra WikiArt-only steps (control)')
    ax.axhline(100 / len(CLASSES), color='#9a9890', lw=1, ls='--')
    ax.text(-0.25, 100 / len(CLASSES) + 0.4, 'chance', fontsize=8, color='#52514e', ha='left', va='bottom')
    ax.set_xticks(list(xs.values()), ['0\n(zero-shot)', '5', '10', '25', '50'])
    ax.set_xlim(-0.3, 4.8)
    ax.set_title(f'Fine-tune on {src.capitalize()}, test on {tgt.capitalize()}', fontsize=11)
    ax.set_xlabel('labeled museum paintings per style')
    ax.grid(axis='y', color='#e6e5e0', lw=0.8); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
axes[0].set_ylabel('balanced accuracy (%), unseen painters')
axes[0].legend(frameon=False, fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig(OUT / 'rq2_curve.png', dpi=200); plt.show()
print('Saved rq2_summary.csv, rq2_report.txt, rq2_curve.png and fewshot_picks.csv to', OUT)